### 1. Бизнес-цели для набора данных для решения задач регрессии и классификации.

Задача регрессии: Предсказать цену автомобиля на основе характеристик. Целевая переменная — Price.

Задача классификации: Отнести автомобиль к определённой ценовой категории. (бюджетный, средний, премиум).

In [1]:
import pandas as pd
import numpy as np

cars = pd.read_csv("car_price_prediction.csv", sep=",")
cars.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19237 entries, 0 to 19236
Data columns (total 18 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   ID                19237 non-null  int64  
 1   Price             19237 non-null  int64  
 2   Levy              19237 non-null  object 
 3   Manufacturer      19237 non-null  object 
 4   Model             19237 non-null  object 
 5   Prod. year        19237 non-null  int64  
 6   Category          19237 non-null  object 
 7   Leather interior  19237 non-null  object 
 8   Fuel type         19237 non-null  object 
 9   Engine volume     19237 non-null  object 
 10  Mileage           19237 non-null  object 
 11  Cylinders         19237 non-null  float64
 12  Gear box type     19237 non-null  object 
 13  Drive wheels      19237 non-null  object 
 14  Doors             19237 non-null  object 
 15  Wheel             19237 non-null  object 
 16  Color             19237 non-null  object

### 2. Выбрать ориентир для каждой задачи.

- Регрессия: Предсказывать среднюю цену автомобиля для всех наблюдений.
- Классификация: Предсказывать самый частый класс автомобилей.

### 3. Определить достижимый уровень качества модели для каждой задачи.

Ожидаемые значения
- Регрессия:
1. R^2 (качество объяснения данных моделью) 0.6 - 0.7
2. MAE (средняя абсолютная ошибка), 2000-5000

- Классификация: 
1. Accuracy. (Аккуратность, доля верных ответов) > 61%
2. F1-score. (Сбалансированность ответов) > 0.70

### 4. Выбрать не менее трех моделей для каждой задачи.

- Регрессия:
1. Linear Regression
2. Random Forest Regressor
3. Gradient Boosting

- Классификация:
1. Logistic Regression
2. Random Forest Classifier
3. Gradient Boosting

### 5. Построить конвейер.

#### Проведем предварительную обработку данных:

In [2]:
# Levy. Преобразование сбора. Заменяем (-) на 0
#cars['Levy'] = cars['Levy'].fillna(0)
cars['Levy'] = pd.to_numeric(cars['Levy'], errors='coerce')
# Заполним пропуски медианой
#cars['Levy'] = cars['Levy'].fillna(cars['Levy'].median())

# Leather interior. Заменим ответы Yes/No на цифры: Yes — 1, No — 0.
cars['Leather interior'] = cars['Leather interior'].map({'Yes': 1, 'No': 0})

# Engine volume + Turbo. Преобразование объема двигателя
# Извлекаем числовое значение и отдельно флаг Turbo
cars['Turbo'] = cars['Engine volume'].str.contains('Turbo').astype(int)
cars['Engine volume'] = cars['Engine volume'].str.extract(r'(\d+\.?\d*)')[0].astype(float)

# Mileage. Преобразование пробега, убираем km, делаем int
cars['Mileage'] = cars['Mileage'].str.replace(r'\D+', '', regex=True).astype(int)

# Doors. Преобразование дверей, в сете все значения вида '02-Mar', имелось в виду 2-3, и вида ">5"
# Заменим такие данные на большее значение
cars['Doors'] = cars['Doors'].replace({
    '02-Mar': 3,
    '04-May': 5,
    '06-Jul': 6, 
    '>5': '5',
})
cars['Doors'] = cars['Doors'].astype(str).str.extract(r'(\d+)')[0].astype(int)

# Wheel. Есть 2 типа: Left wheel и Right-hand drive, заменим их бинарно
cars['Wheel'] = cars['Wheel'].map({'Left wheel': 0, 'Right-hand drive': 1})

# добавляем столбец с возрастом автомобиля
cars['Age'] = 2025 - cars['Prod. year']
# удляем признак Prod. year
cars.drop(columns='Prod. year', inplace=True)

cars.info()
cars.describe().T
cars.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19237 entries, 0 to 19236
Data columns (total 19 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   ID                19237 non-null  int64  
 1   Price             19237 non-null  int64  
 2   Levy              13418 non-null  float64
 3   Manufacturer      19237 non-null  object 
 4   Model             19237 non-null  object 
 5   Category          19237 non-null  object 
 6   Leather interior  19237 non-null  int64  
 7   Fuel type         19237 non-null  object 
 8   Engine volume     19237 non-null  float64
 9   Mileage           19237 non-null  int64  
 10  Cylinders         19237 non-null  float64
 11  Gear box type     19237 non-null  object 
 12  Drive wheels      19237 non-null  object 
 13  Doors             19237 non-null  int64  
 14  Wheel             19237 non-null  int64  
 15  Color             19237 non-null  object 
 16  Airbags           19237 non-null  int64 

,ID,Price,Levy,Manufacturer,Model,Category,Leather interior,Fuel type,Engine volume,Mileage,Cylinders,Gear box type,Drive wheels,Doors,Wheel,Color,Airbags,Turbo,Age
0,45654403,13328,1399.0,LEXUS,RX 450,Jeep,1,Hybrid,3.5,186005,6.0,Automatic,4x4,5,0,Silver,12,0,15
1,44731507,16621,1018.0,CHEVROLET,Equinox,Jeep,0,Petrol,3.0,192000,6.0,Tiptronic,4x4,5,0,Black,8,0,14
2,45774419,8467,NaN,HONDA,FIT,Hatchback,0,Petrol,1.3,200000,4.0,Variator,Front,5,1,Black,2,0,19
3,45769185,3607,862.0,FORD,Escape,Jeep,1,Hybrid,2.5,168966,4.0,Automatic,4x4,5,0,White,0,0,14
4,45809263,11726,446.0,HONDA,FIT,Hatchback,1,Petrol,1.3,91901,4.0,Automatic,Front,5,0,Silver,4,0,11


#### Разделяем данные до обработки

In [3]:
from sklearn.model_selection import train_test_split

# Целевые переменные

# Для регрессии
X = cars.drop(columns=['Price', 'ID'])
y = cars['Price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Для классификации. Разделим Price на 3 группы по квантилям
cars['PriceCategory'] = pd.qcut(cars['Price'], q=3, labels=['budget', 'medium', 'premium'])

# еще плюсом прайс категории для классификации
y_clf = cars['PriceCategory']
# без айди, цены и прайс категории
X_clf = cars.drop(columns=['Price', 'PriceCategory', 'ID'])

X_train_clf, X_test_clf, y_train_clf, y_test_clf = train_test_split(X_clf, y_clf, test_size=0.2, random_state=42, stratify=y_clf)

display("Обучающая выборка:", X_train.shape)
display("Тестовая выборка:", X_test.shape)
print('--------------')
display("Обучающая выборка:", X_train_clf.shape)
display("Тестовая выборка:", X_test_clf.shape)

'Обучающая выборка:'

(15389, 17)

'Тестовая выборка:'

(3848, 17)

--------------


'Обучающая выборка:'

(15389, 17)

'Тестовая выборка:'

(3848, 17)

#### 5.2 Кодирование категориальных признаков:

In [4]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Определение признаков. Числовые и категориальные
numeric_features = ['Levy', 'Engine volume', 'Mileage', 'Cylinders', 'Doors', 'Airbags', 'Age']
categorical_features = ['Manufacturer', 'Model', 'Category', 'Fuel type', 'Gear box type', 'Drive wheels', 'Color']

# Обработка числовых данных
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),  # Заполнение пропусков медианой
    ('scaler', StandardScaler())                    # Нормализация данных
])

# Обработка категориальных данных
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),  # Заполнение пропусков модой
    ('onehot', OneHotEncoder(handle_unknown='ignore'))     # Преобразование в One-Hot Encoding
])

# Объединяем в ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),   # Применяем числовую обработку
        ('cat', categorical_transformer, categorical_features)  # Применяем категориальную обработку
    ]
)

### 6. Реализовать подбор гиперпараметров для каждой модели

In [5]:
from sklearn.linear_model import Ridge
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

# Создаем пайплайн, куда входят предобработка и модель
def make_pipeline(model):
    return Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('model', model)
    ])

# --- 1) Ridge Regression ---
ridge_pipeline = make_pipeline(Ridge(random_state=42))
# Параметры для Ridge Regression
ridge_params = {
    'model__alpha': [0.1, 1.0, 10.0, 100.0],  # Основной параметр регуляризации
    'model__fit_intercept': [True],
}

ridge_grid = GridSearchCV(
    ridge_pipeline, 
    ridge_params, 
    cv=3, 
    n_jobs=-1, 
    scoring='r2',
    verbose=0 
)

ridge_grid.fit(X_train, y_train)

print("Лучшие параметры Ridge Regression:", ridge_grid.best_params_)

Лучшие параметры Ridge Regression: {'model__alpha': 100.0, 'model__fit_intercept': True}


In [6]:
rf_pipeline = make_pipeline(RandomForestRegressor(random_state=42))
rf_params = {
    'model__n_estimators': [100, 200],
    'model__max_depth': [10, 20, None],
    'model__min_samples_split': [2, 5],
}

rf_grid = GridSearchCV(rf_pipeline, rf_params, cv=3, n_jobs=-1, scoring='r2')
rf_grid.fit(X_train, y_train)

print("Лучшие параметры RandomForest:", rf_grid.best_params_)

Лучшие параметры RandomForest: {'model__max_depth': None, 'model__min_samples_split': 2, 'model__n_estimators': 200}


In [7]:
gb_pipeline = make_pipeline(GradientBoostingRegressor(random_state=42))
gb_params = {
    'model__n_estimators': [100, 200],
    'model__learning_rate': [0.05, 0.1],
    'model__max_depth': [3, 4],
}

gb_grid = GridSearchCV(gb_pipeline, gb_params, cv=3, n_jobs=-1, scoring='r2')
gb_grid.fit(X_train, y_train)

print("Лучшие параметры GradientBoosting:", gb_grid.best_params_)

Лучшие параметры GradientBoosting: {'model__learning_rate': 0.05, 'model__max_depth': 3, 'model__n_estimators': 100}


### Классификация

In [8]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

logreg_pipeline = make_pipeline(LogisticRegression(max_iter=2000))
logreg_params = {
    'model__C': [0.1, 1, 10]
}

logreg_grid = GridSearchCV(logreg_pipeline, logreg_params, cv=3, n_jobs=-1, scoring='accuracy')
logreg_grid.fit(X_train_clf, y_train_clf)

print("Лучшие параметры Logistic Regression:", logreg_grid.best_params_)

Лучшие параметры Logistic Regression: {'model__C': 10}


In [9]:
rf_clf_pipeline = make_pipeline(RandomForestClassifier(random_state=42))
rf_clf_params = {
    'model__n_estimators': [100, 200],
    'model__max_depth': [10, 20, None]
}

rf_clf_grid = GridSearchCV(rf_clf_pipeline, rf_clf_params, cv=3, n_jobs=-1, scoring='accuracy')
rf_clf_grid.fit(X_train_clf, y_train_clf)

print("Лучшие параметры RandomForest Classifier:", rf_clf_grid.best_params_)

Лучшие параметры RandomForest Classifier: {'model__max_depth': None, 'model__n_estimators': 200}


In [10]:
gb_clf_pipeline = make_pipeline(GradientBoostingClassifier(random_state=42))
gb_clf_params = {
    'model__n_estimators': [100, 200],
    'model__learning_rate': [0.05, 0.1]
}

gb_clf_grid = GridSearchCV(gb_clf_pipeline, gb_clf_params, cv=3, n_jobs=-1, scoring='accuracy')
gb_clf_grid.fit(X_train_clf, y_train_clf)

print("Лучшие параметры GradientBoostingClassifier:", gb_clf_grid.best_params_)

Лучшие параметры GradientBoostingClassifier: {'model__learning_rate': 0.1, 'model__n_estimators': 200}


### 7. Обучить модели

#### Регрессия

In [11]:
best_ridge = ridge_grid.best_estimator_
best_rf = rf_grid.best_estimator_
best_gb = gb_grid.best_estimator_

#### Классификация

In [12]:
best_logreg = logreg_grid.best_estimator_
best_rf_clf = rf_clf_grid.best_estimator_
best_gb_clf = gb_clf_grid.best_estimator_

### 8. Оценить качество моделей для решения каждой из задач

#### Регрессия

In [13]:
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

models = {
    "Ridge Regression": best_ridge,
    "Random Forest": best_rf,
    "Gradient Boosting": best_gb
}

for name, model in models.items():
    y_pred = model.predict(X_test)
    print(f"\n--- {name} ---")
    print("R2:", r2_score(y_test, y_pred))
    print("MAE:", mean_absolute_error(y_test, y_pred))
    print("MSE:", mean_squared_error(y_test, y_pred))


--- Ridge Regression ---
R2: -0.7506824476446239
MAE: 13220.384345787024
MSE: 545506724.0695589

--- Random Forest ---
R2: 0.6603311749322112
MAE: 4408.693460861526
MSE: 105839656.00419293

--- Gradient Boosting ---
R2: 0.3904688928447234
MAE: 8791.012788080734
MSE: 189927829.53305876


#### Классификация

In [14]:
from sklearn.metrics import accuracy_score, f1_score, classification_report

clf_models = {
    "Logistic Regression": best_logreg,
    "Random Forest Classifier": best_rf_clf,
    "Gradient Boosting Classifier": best_gb_clf
}

for name, model in clf_models.items():
    y_pred = model.predict(X_test_clf)
    print(f"\n--- {name} ---")
    print("Accuracy:", accuracy_score(y_test_clf, y_pred))
    print("F1 macro:", f1_score(y_test_clf, y_pred, average='macro'))
    print("\nClassification Report:\n", classification_report(y_test_clf, y_pred))


--- Logistic Regression ---
Accuracy: 0.6782744282744283
F1 macro: 0.6771985649859339

Classification Report:
               precision    recall  f1-score   support

      budget       0.66      0.72      0.69      1283
      medium       0.66      0.60      0.63      1283
     premium       0.71      0.72      0.72      1282

    accuracy                           0.68      3848
   macro avg       0.68      0.68      0.68      3848
weighted avg       0.68      0.68      0.68      3848


--- Random Forest Classifier ---
Accuracy: 0.8027546777546778
F1 macro: 0.8031202731942084

Classification Report:
               precision    recall  f1-score   support

      budget       0.84      0.82      0.83      1283
      medium       0.74      0.76      0.75      1283
     premium       0.82      0.82      0.82      1282

    accuracy                           0.80      3848
   macro avg       0.80      0.80      0.80      3848
weighted avg       0.80      0.80      0.80      3848


--- Grad

#### Вывод:
- Регрессия:
Критерии: коэффициент детерминации R^2, R^2 = 1: идеальное предсказание. R^2 = 0: модель не лучше, чем предсказание среднего значения. R^2 < 0: модель хуже, чем просто предсказание среднего.
MAE (Mean Absolute Error). Средняя абсолютная ошибка в тех же единицах, что и Price, чем меньше значение — тем лучше.

1. Ridge Regression: R^2 = -0.75. показала очень плохой результат, это можно объяснить тем, что для Model создаются сотни бинарных признаков, из-за этого линейная модель с регуляризацией не справляется с такой структурой.
2. Random Forest: R^2 = 0.66. Уже лучше результат, модель может объяснить 66%. MAE = 4400, это ошибка предсказания. Среди всех моделей, это лучший результат.
3. Gradient Boosting: R^2 = 0.39. Результат хуже, чем у Random Forest. MAE = 8800, этов 2 раза хуже, чем у Random Forest. Возможно, это произошло из-за недостаточной настройки гиперпараметров и наличия шума.
   
- Классификация:
Критерии: Accuracy — Доля правильно предсказанных объектов среди всех. Хороша, если классы сбалансированы. F1 — Среднее F1-меры по всем классам.

1. Logistic Regression: Accuracy = 67.8%, F1 = 67.7%. Модель немного лучше случайного угадывания.
2. Random Forest Classifier: Accuracy = 80.3%, F1 = 80.3%. Лучшая модель для классификации, хорошо работает с категориальными признаками и устойчива к шуму.
3. Gradient Boosting Classifier: Accuracy = 74.4%, F1 = 74.3%. Высокий recall для budget (0.83) это значит, что почти все бюджетные авто найдены, но низкий recall для premium (0.73) это значит, что 27% премиум-авто ошибочно отнесены к другим категориям.